# Dual-SR830 commissioning sweep browser

Read-only analysis for standalone frequency and excitation sweep JSON/JSONL files. Completed records and clean formal samples are the defaults. Rejected records require explicit audit mode; transition and cleanup payloads are never mixed into formal curves.

In [ ]:
from pathlib import Path
from IPython.display import display

from attodry_control.commissioning_analysis import (
    browse_and_load_commissioning_file,
    discover_commissioning_records,
    export_commissioning_csv,
    load_sweep_samples,
    plot_commissioning_sweep,
    summarize_commissioning_file,
)

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory.parent
    if working_directory.name.lower() == 'notebooks'
    else working_directory
)
DATA_DIRECTORY = PROJECT_ROOT / 'run_data' / 'commissioning'
DATA_DIRECTORY

## Catalog and record-status filters

Available record statuses are `completed`, `rejected`, `diagnostic`, `other`, and `invalid`. Keep `completed` alone for ordinary plotting. Add `rejected` only for explicit audit work.

In [ ]:
RECORD_STATUSES = {'completed'}
SCAN_TYPES = {'frequency', 'excitation'}

catalog = discover_commissioning_records(
    DATA_DIRECTORY,
    record_statuses=RECORD_STATUSES,
    scan_types=SCAN_TYPES,
)
catalog_view = [
    {
        'file': item.path.name,
        'scan': item.scan_type,
        'status': item.record_status,
        'samples': item.sample_count,
        'problem_samples': item.problem_count,
        'error': item.error,
    }
    for item in catalog
]
catalog_view

## Browse and open a file directly

Set `OPEN_BROWSER=True` and rerun the cell to open the native Windows file dialog. If the dialog is unavailable in a remote Jupyter session, assign `selected_path = Path(...)` directly.

In [ ]:
OPEN_BROWSER = False
browsed = (
    browse_and_load_commissioning_file(DATA_DIRECTORY)
    if OPEN_BROWSER
    else None
)
if browsed is not None:
    selected_path, selected_raw_data = browsed
    selected_summary = summarize_commissioning_file(selected_path)
    display(selected_summary)


## Select the newest completed frequency and excitation records

The catalog is newest-first. You may replace either path with a browsed path.

In [ ]:
completed_catalog = discover_commissioning_records(
    DATA_DIRECTORY, record_statuses={'completed'}
)
frequency_record = next(
    (item for item in completed_catalog if item.scan_type == 'frequency'), None
)
excitation_record = next(
    (item for item in completed_catalog if item.scan_type == 'excitation'), None
)
if frequency_record is None or excitation_record is None:
    raise FileNotFoundError('A completed frequency or excitation record is missing.')

frequency_path = frequency_record.path
excitation_path = excitation_record.path
frequency_path, excitation_path

## Formal-sample status filters

Available sample statuses are `clean`, `problem`, `unlocked`, `overload`, and `instrument_error`. `clean` is the publication default. Set `SAMPLE_STATUSES=None` to inspect every formal sample. To load a rejected record, also set `INCLUDE_REJECTED=True`; that opt-in is never automatic.

In [ ]:
SAMPLE_STATUSES = {'clean'}
ROLES = {'xx', 'xy'}
INCLUDE_REJECTED = False

frequency_rows = load_sweep_samples(
    frequency_path,
    include_rejected=INCLUDE_REJECTED,
    sample_statuses=SAMPLE_STATUSES,
    roles=ROLES,
)
excitation_rows = load_sweep_samples(
    excitation_path,
    include_rejected=INCLUDE_REJECTED,
    sample_statuses=SAMPLE_STATUSES,
    roles=ROLES,
)
selected_rows = (
    load_sweep_samples(
        selected_path,
        include_rejected=INCLUDE_REJECTED,
        sample_statuses=SAMPLE_STATUSES,
        roles=ROLES,
    )
    if browsed is not None and selected_summary.scan_type in {'frequency', 'excitation'}
    else ()
)
len(frequency_rows), len(excitation_rows), len(selected_rows), frequency_rows[:2]

## Plot frequency and source-amplitude sweeps

Each point shows the mean and sample standard deviation of the selected formal samples. Frequency uses a logarithmic x axis; excitation uses source voltage by default and can be changed to `nominal_current_a_rms`.

In [ ]:
for metric in ('x_v', 'y_v', 'amplitude_v', 'phase_deg'):
    display(plot_commissioning_sweep(frequency_rows, metric=metric))

for metric in ('x_v', 'y_v', 'amplitude_v', 'phase_deg'):
    display(
        plot_commissioning_sweep(
            excitation_rows, metric=metric, x_axis='source_v_rms', log_x=False
        )
    )

if selected_rows:
    display(plot_commissioning_sweep(selected_rows, metric='amplitude_v'))


## Optional CSV and figure export

No files are written unless `SAVE_OUTPUTS=True`.

In [ ]:
SAVE_OUTPUTS = False
OUTPUT_DIRECTORY = PROJECT_ROOT / 'analysis_output' / 'sr830_commissioning'
if SAVE_OUTPUTS:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    export_commissioning_csv(
        frequency_rows, OUTPUT_DIRECTORY / 'frequency_samples.csv'
    )
    export_commissioning_csv(
        excitation_rows, OUTPUT_DIRECTORY / 'excitation_samples.csv'
    )
    for scan_name, rows in (
        ('frequency', frequency_rows),
        ('excitation', excitation_rows),
    ):
        for metric in ('x_v', 'y_v', 'amplitude_v', 'phase_deg'):
            figure = plot_commissioning_sweep(rows, metric=metric)
            figure.savefig(OUTPUT_DIRECTORY / f'{scan_name}_{metric}.png', dpi=200)
            figure.savefig(OUTPUT_DIRECTORY / f'{scan_name}_{metric}.pdf')
    display(OUTPUT_DIRECTORY)
